## Ensemble_Agent Experimenting

In [ ]:
from deal_hunter.agents.items import Item

#sys
from pathlib import Path
import sys
root_dir = Path.cwd().resolve().parents[0]
sys.path.insert(0,str(root_dir/"src"))




/Volumes/256 SSD/deal_hunter/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
#imports
import os 
from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.auto import tqdm



### Load Dataset

In [6]:
#hf login
load_dotenv(override = True)
hf_token = os.environ["HF_TOKEN"]
login(token=hf_token , add_to_git_credential = False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
pricer_data = "Vishy08/pricer-data"
train , test, val = Item.from_hub(dataset_name=pricer_data)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 320,000 training items, 4,000 validation items, 8,000 test items


In [8]:
test[0]

Item(title='ZLINE 30 in. Wooden Wall Mount Range Hood in Cottage White - Includes Motor (KPTT-30)', price=899.95, category='Appliances', test_prompt="How much does this cost to the nearest dollar?\n\nZLINE 30 in. Wooden Wall Mount Range Hood in Cottage White - Includes Motor\nThe ZLINE KPTT wall mount wooden range hood designed to be both elegant and powerful, featuring the industry's only lifetime warranty motor. The hand-finished painted cottage white wood is made from solid pine with a stainless steel inner frame, making it both durable and long-lasting. This durable construction, along with ZLINE's lifetime warranty on the motor, guarantees a range hood that will last for a lifetime. This classic wooden hood contains many modern features, such as Hand-carved, hand-finished wood; Dishwasher-safe stainless steel baffle filters; Built-in LED lighting; High performance motor with speeds up to 400 CFM. All ZLINE range hoods come equipped with everything needed to easily install and use,

### Testing ChromaDB

In [9]:
import chromadb
DB = "products_vectorstore"
chroma_client = chromadb.PersistentClient(path = DB)

In [10]:
import chromadb
chroma_client = chromadb.Client()

# switch \`create_collection\` to \`get_or_create_collection\` to avoid creating a new collection every time
collection = chroma_client.get_or_create_collection(name="my_collection")

# switch \`add\` to \`upsert\` to avoid adding the same documents every time
collection.upsert(
    documents=[
        "This is a document about pineapple",
        "This is a document about oranges"
    ],
    ids=["id1", "id2"]
)

results = collection.query(
    query_texts=["This is a query document about florida"], # Chroma will embed this for you
    n_results=2 # how many results to return
)

print(results)

{'ids': [['id2', 'id1']], 'embeddings': None, 'documents': [['This is a document about oranges', 'This is a document about pineapple']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[1.1462141275405884, 1.3015384674072266]]}


### Encoder LLM (all-Mini-L6-v2)

In [11]:
#Using SentenceTransformer Encoding LLM ( Free(fast) + Private)
from sentence_transformers import SentenceTransformer
encoder_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8144.89it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
sentences = [
    item.test_prompt for item in test[:50]
]
embeddings = encoder_model.encode(sentences=sentences)


In [13]:
type(embeddings)

numpy.ndarray

### Building ChromaDB 

In [14]:
def summarizer(item):
    question = "How much does this cost to the nearest dollar?\n\n"
    prefix = "\n\nPrice is $"
    summary = item.test_prompt.replace(question,"").replace(prefix,"")
    return f"{summary}"

In [15]:
summarizer(test[3000])

'POWERTEC 70107V 4 T-Fitting for Dust Collection Hose, 1 PK\nIntroducing the 4-Inch T-Fitting by POWERTEC. This handy t-shaped connector optimizes the performance of your dust collection system by allowing you to add branches (connections) to your setup without compromising its efficiency. This creates a secure, airtight connection to ensure that harmful dust, dirt and debris is sucked in and expelled directly into your dust collector – resulting in clean air and a clean work environment. This dust collector hose adapter is ideal for professionals, hobbyists and workshops where space constraints or machine design are a concern. T-Shaped Fitting Dimensions 4-Inch OD nominal ends measure O.D. and I.D. Premium Build The POWERTEC T-Fitting is made out of high quality ABS plastic. This lightweight material is commonly known for it’s super durability'

In [16]:
def build_product_collection(chroma_client , encoder_model, items, name="products", batch_size=1000):

    collection = chroma_client.get_or_create_collection(name) 
    
    # processing data in batch of 1000 
    with tqdm(total = len(items) , desc=f"Chroma {name}:", unit = "doc",dynamic_ncols = True) as pbar:
        for start in range(0,len(items),batch_size):
            #creating the batch
            batch = items[start : start + batch_size] 
            
            #include title + description 
            docs = [summarizer(item) for item in batch]
            embeddings = encoder_model.encode(docs).astype(float).tolist()

            metas = [{"category" : item.category , "price" : item.price} for item in batch]
            doc_ids = [f"{name}_{start + k}" for k in range(len(batch))]

            collection.upsert(
                documents = docs,
                metadatas =metas,
                ids = doc_ids,
                embeddings = embeddings
            )
            pbar.update(len(batch))
    
    return collection


In [17]:
collection = build_product_collection(chroma_client, encoder_model, train[:])


Chroma products:: 100%|██████████| 320000/320000 [46:57<00:00, 113.59doc/s]


### Running and Similarity Search

In [18]:
chroma_client.get_collection("products")

Collection(name=products)

In [19]:
text = ["POWERTEC 70107V 4 T-Fitting for Dust Collection Hose, 1 PK\nIntroducing the 4-Inch T-Fitting by POWERTEC. This handy t-shaped connector optimizes the performance of your dust collection system by allowing you to add branches (connections) to your setup without compromising its efficiency. This creates a secure, airtight connection to ensure that harmful dust, dirt and debris is sucked in and expelled directly into your dust collector – resulting in clean air and a clean work environment. This dust collector hose adapter is ideal for professionals, hobbyists and workshops where space constraints or machine design are a concern. T-Shaped Fitting Dimensions 4-Inch OD nominal ends measure O.D. and I.D. Premium Build The POWERTEC T-Fitting is made out of high quality ABS plastic. This lightweight material is commonly known for it’s super durability"]
result = collection.query( query_texts=text,n_results = 3)
print(result["documents"][0][:])

['POWERTEC 70125 4-Inch Pipe Hose Adapter - ABS Plastic Quick Coupler for Workshop Dust Collection Fittings, Black\nThe POWERTEC Hose Adapter, is designed to make connections, easily and securely from your pipe to a multitude of fittings; narrow end fits into ordinary 4-Inch PVC pipes (for example, drain, waste, vent, etc), and tapered end with 4-Inch ID fits most 4-Inch dust collection fittings( wyes, tees, elbow and blast gates, etc). Ensuring airtight hose to fitting linkage, this handy device will be a strong link in your dust-collection system. INCLUDES (1) Pipe Hose (Quick Coupler) Adapter FUNCTION Makes a quick and easy connection from your pipe to a multitude of 4” dust collector fittings – Takes all of the work out of connecting, disconnecting or switching from', 'POWERTEC 2-1/2 to 1-1/4 Hose Reducer, 2 PK\nIntroducing the 2.5 to 1.25 Hose Reducer by POWERTEC. This adapter serves as a conversion unit for your dust collection accessories and attachments. Designed for profession

In [20]:
#creating a function to find similar and convert the given text to embedding

def find_similar(text):
    vector = encoder_model.encode(text) 
    result = collection.query(query_embeddings=vector,n_results=5)
    return result["documents"][0][:]


In [ ]:
find_similar(text)

['POWERTEC 70125 4-Inch Pipe Hose Adapter - ABS Plastic Quick Coupler for Workshop Dust Collection Fittings, Black\nThe POWERTEC Hose Adapter, is designed to make connections, easily and securely from your pipe to a multitude of fittings; narrow end fits into ordinary 4-Inch PVC pipes (for example, drain, waste, vent, etc), and tapered end with 4-Inch ID fits most 4-Inch dust collection fittings( wyes, tees, elbow and blast gates, etc). Ensuring airtight hose to fitting linkage, this handy device will be a strong link in your dust-collection system. INCLUDES (1) Pipe Hose (Quick Coupler) Adapter FUNCTION Makes a quick and easy connection from your pipe to a multitude of 4” dust collector fittings – Takes all of the work out of connecting, disconnecting or switching from',
 'POWERTEC 2-1/2 to 1-1/4 Hose Reducer, 2 PK\nIntroducing the 2.5 to 1.25 Hose Reducer by POWERTEC. This adapter serves as a conversion unit for your dust collection accessories and attachments. Designed for professio